In [7]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf 
from lsdb import read_hats
import matplotlib.pyplot as plt
from dask.distributed import Client
from nested_pandas import NestedDtype

In [3]:
def test_lc(path):

    LC_COLUMN = "lc"

    catalog = read_hats(
        path,    
    ).map_partitions(
        lambda df: df.assign(
            **{LC_COLUMN: df[LC_COLUMN].astype(NestedDtype.from_pandas_arrow_dtype(df.dtypes[LC_COLUMN]))},
        )
    )

    with Client() as client:
        # display(client)
        df = catalog.compute()
    
    return df
path='/media3/majumder/dataset/cepheids/hats/zubercal_vcep'
df = test_lc(path)

In [ ]:

id_=1085440417799958669
data = pd.DataFrame(df[df.index==id_]["lc"].values[0])
data["band"].replace({"i": "#e8ce3c", "r": "#297560", "g": "#221442"}, inplace=True)

In [ ]:
count=0
counter = 600
filenames = list()
path = f"/media3/majumder/dataset/cepheids/test/"
#
#
#
for p in os.listdir(path):
    for lbl in os.listdir(path+p):
        for cnk in os.listdir(path+p+"/"+lbl):
            filenames.append(path+p+"/"+lbl+'/'+cnk)
#
#
#
dataset = tf.data.TFRecordDataset(filenames)
#
#
#
for rec in dataset:
    #
    example = tf.train.SequenceExample()
    example.ParseFromString(rec.numpy())
    #
    last_index = example.context.feature['last_index'].int64_list.value
    time = example.feature_lists.feature_list['dim_0'].feature[0].float_list.value
    mag = example.feature_lists.feature_list['dim_1'].feature[0].float_list.value
    band_sorted = example.feature_lists.feature_list['dim_3'].feature[0].float_list.value
    last_index = example.context.feature['last_index'].int64_list.value
    id_ = example.context.feature['id'].int64_list.value[0]
    if count== counter:
        print("ID: ", id_)
        print("Last index: ", last_index)
        break
    count += 1

In [ ]:
plt.figure(figsize=(12, 6))
plt.suptitle(f"ID: {id_}")
plt.subplot(1, 2, 1)
plt.scatter(data["mjd"], data["mag"], c=data["band"], s=10)
plt.subplot(1, 2, 2)
plt.scatter(time, mag, c=band_sorted, s=10)